In [3]:
# Import libraries for data handling, visualization, text processing, similarity calculation, and saving the model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [4]:
# Load the movies dataset from a CSV file into a DataFrame
movies = pd.read_csv("../Netfilx_Recommendation/Data/top10K-TMDB-movies.csv", engine='python')

In [5]:
# Display the first few rows of the dataset to preview the data
movies.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [6]:
# Show number of rows and columns in the dataset
movies.shape

(10000, 9)

In [7]:
# Display column names, data types, and non-null values
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10000 non-null  int64  
 1   title              10000 non-null  object 
 2   genre              9997 non-null   object 
 3   original_language  10000 non-null  object 
 4   overview           9987 non-null   object 
 5   popularity         10000 non-null  float64
 6   release_date       10000 non-null  object 
 7   vote_average       10000 non-null  float64
 8   vote_count         10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB


In [8]:
# Show basic statistics for numerical columns
movies.describe()

,id,popularity,vote_average,vote_count
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,161243.505000,34.697267,6.621150,1547.309400
std,211422.046043,211.684175,0.766231,2648.295789
min,5.000000,0.600000,4.600000,200.000000
25%,10127.750000,9.154750,6.100000,315.000000
50%,30002.500000,13.637500,6.600000,583.500000
75%,310133.500000,25.651250,7.200000,1460.000000
max,934761.000000,10436.917000,8.700000,31917.000000


In [9]:
# Select the 'title' column, take the first 50 entries, and return only the unique titles
movies["title"][:50].unique()

array(['The Shawshank Redemption', 'Dilwale Dulhania Le Jayenge',
       'The Godfather', "Schindler's List", 'The Godfather: Part II',
       'Impossible Things', 'Spirited Away', 'Your Eyes Tell',
       'Dou kyu sei – Classmates', 'Your Name.', '12 Angry Men',
       "Gabriel's Inferno", 'Parasite', 'The Green Mile',
       "Gabriel's Inferno: Part II", 'The Dark Knight',
       'The Good, the Bad and the Ugly', 'Pulp Fiction',
       'The Lord of the Rings: The Return of the King',
       "Gabriel's Inferno: Part III", 'Forrest Gump', 'Cinema Paradiso',
       'Seven Samurai', 'GoodFellas', 'Violet Evergarden: The Movie',
       'Life Is Beautiful', 'Once Upon a Time in America', 'Harakiri',
       'Psycho', 'Josee, the Tiger and the Fish', "A Dog's Will",
       'Grave of the Fireflies', "One Flew Over the Cuckoo's Nest",
       'Fight Club', 'Evangelion: 3.0+1.0 Thrice Upon a Time',
       'Spider-Man: Into the Spider-Verse', 'City of God', 'Hope',
       'A Silent Voice: The Mov

In [10]:
# Count how many times each movie title appears
movies["title"].value_counts()

title
Beauty and the Beast                             4
Godzilla                                         3
Taxi                                             3
The Three Musketeers                             3
War of the Buttons                               3
                                                ..
Anplagghed al cinema                             1
Another Year                                     1
Fish Tank                                        1
The Tin Drum                                     1
In the Name of the King: A Dungeon Siege Tale    1
Name: count, Length: 9661, dtype: int64

In [11]:
# Get all unique genres in the dataset
movies["genre"].unique()

array(['Drama,Crime', 'Comedy,Drama,Romance', 'Drama,History,War', ...,
       'Action,TV Movie,Science Fiction,Comedy,Adventure',
       'Action,Science Fiction,War', 'Adventure,Fantasy,Action,Drama'],
      dtype=object)

In [12]:
# Count how many movies belong to each genre
movies["genre"].value_counts()

genre
Comedy                                    744
Drama                                     611
Drama,Romance                             290
Comedy,Drama                              262
Comedy,Romance                            255
                                         ... 
Fantasy,Animation,Romance,Family            1
Drama,Thriller,Crime,Western                1
Comedy,Drama,Romance,Fantasy,Adventure      1
Drama,History,Action                        1
Adventure,Fantasy,Action,Drama              1
Name: count, Length: 2123, dtype: int64

In [13]:
# Show only the 'title' and 'popularity' columns
movies[["title","popularity"]]

,title,popularity
0,The Shawshank Redemption,94.075
1,Dilwale Dulhania Le Jayenge,25.408
2,The Godfather,90.585
3,Schindler's List,44.761
4,The Godfather: Part II,57.749
...,...,...
9995,The Last Airbender,98.322
9996,Sharknado 3: Oh Hell No!,12.490
9997,Captain America,18.333
9998,In the Name of the King: A Dungeon Siege Tale,15.159


In [14]:
# Count how many missing values are in each column
movies.isna().sum()


id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

In [15]:
# Count how many duplicate rows are in the dataset
movies.duplicated().sum()


0

In [16]:
# Show all column names in the dataset
movies.columns

Index(['id', 'title', 'genre', 'original_language', 'overview', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

In [17]:
# Keep only the 'id', 'title', 'genre', and 'overview' columns
movies=movies[['id', 'title', 'genre', 'overview']]
movies

,id,title,genre,overview
0,278,The Shawshank Redemption,"Drama,Crime",Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Drama,Crime","Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,"Drama,History,War",The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,"Drama,Crime",In the continuing saga of the Corleone crime f...
...,...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy","The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"Action,Science Fiction,War","During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",A man named Farmer sets out to rescue his kidn...


In [18]:
# Create a new column 'tags' by combining 'overview' and 'genre'
movies["tags"]=movies["overview"]+movies["genre"]
movies

,id,title,genre,overview,tags
0,278,The Shawshank Redemption,"Drama,Crime",Framed in the 1940s for the double murder of h...,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second...","Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Drama,Crime","Spanning the years 1945 to 1955, a chronicle o...","Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,"Drama,History,War",The true story of how businessman Oskar Schind...,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,"Drama,Crime",In the continuing saga of the Corleone crime f...,In the continuing saga of the Corleone crime f...
...,...,...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy","The story follows the adventures of Aang, a yo...","The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",The sharks take bite out of the East Coast whe...,The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"Action,Science Fiction,War","During World War II, a brave, patriotic Americ...","During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",A man named Farmer sets out to rescue his kidn...,A man named Farmer sets out to rescue his kidn...


In [19]:
# Create a new dataframe by dropping the 'genre' and 'overview' columns
new_data=movies.drop(columns=['genre','overview'])
new_data

,id,title,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...
...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...


In [20]:
# Create a CountVectorizer with max 10000 features and English stop words
cv=CountVectorizer(max_features=10000,stop_words='english')
cv

CountVectorizer(max_features=10000, stop_words='english')

In [21]:
# Convert the 'tags' column to a numeric feature matrix
vector=cv.fit_transform(new_data['tags'].values.astype('U')).toarray()

# Check the shape of the resulting matrix
vector.shape

(10000, 10000)

In [22]:
# Calculate cosine similarity between all movies
similarity=cosine_similarity(vector)
similarity

array([[1.        , 0.05634362, 0.12888482, ..., 0.07559289, 0.11065667,
        0.06388766],
       [0.05634362, 1.        , 0.07624929, ..., 0.        , 0.03636965,
        0.        ],
       [0.12888482, 0.07624929, 1.        , ..., 0.02273314, 0.06655583,
        0.08645856],
       ...,
       [0.07559289, 0.        , 0.02273314, ..., 1.        , 0.03253   ,
        0.02817181],
       [0.11065667, 0.03636965, 0.06655583, ..., 0.03253   , 1.        ,
        0.0412393 ],
       [0.06388766, 0.        , 0.08645856, ..., 0.02817181, 0.0412393 ,
        1.        ]])

In [23]:
# Find the index of the movie titled "The Godfather"
new_data[new_data['title'] == "The Godfather"].index[0]

# Get a list of all movies sorted by similarity to the movie at index 2 (The Godfather)
distance = sorted(list(enumerate(similarity[2])), reverse=True, key=lambda vector: vector[1])

# Print the titles of the top 10 most similar movies
for i in distance[0:10]:
    print(new_data.iloc[i[0]].title)

The Godfather
The Godfather: Part II
Blood Ties
Joker
Bomb City
Gotti
Felon
Rope
Batman: The Killing Joke
The Big Heat


In [24]:
# This function takes a movie title and prints the top 10 most similar movies based on their 'tags'
def Recommend(movies):
    index=new_data[new_data['title']==movies].index[0]
    distance=sorted(list(enumerate(similarity[index])),reverse=True ,key=lambda vector:vector[1])
    for i in distance[0:10]:
        print(new_data.iloc[i[0]].title)

In [25]:
# Call the Recommend function to print the top 10 movies similar to "The Godfather"
Recommend("The Godfather")

The Godfather
The Godfather: Part II
Blood Ties
Joker
Bomb City
Gotti
Felon
Rope
Batman: The Killing Joke
The Big Heat


In [26]:
# Save the processed movies dataframe to a file using pickle
pickle.dump(new_data, open('movies_list.pkl', 'wb'))

# Save the similarity matrix to a file using pickle
pickle.dump(similarity, open('similarity.pkl', 'wb'))
